#### Magnitude Confidence Model

The magnitude-confidence model estimates the reliability of each predicted overnight
return magnitude.

The primary magnitude model predicts the size of the overnight move. A separate model
then predicts the expected absolute error of that magnitude forecast:

\[
\text{magnitude error}
=
|\text{predicted magnitude} - \text{actual magnitude}|
\]

A lower expected error implies higher magnitude confidence.

The validation period is divided chronologically:

- Early validation: train the error model.
- Late validation: evaluate confidence quality.

The test split remains untouched.

In [36]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import lightgbm as lgb

from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error

SEED = 42

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = (
    PROJECT_ROOT / "data" / "processed"
)

VALIDATION_PREDICTIONS_PATH = (
    PROCESSED_DATA_DIR
    / "magnitude_validation_predictions.parquet"
)

MODEL_PANEL_PATH = (
    PROCESSED_DATA_DIR
    / "magnitude_model_panel.parquet"
)

print("Processed directory:", PROCESSED_DATA_DIR)

Processed directory: /Users/kushagr/Desktop/astra-assignment/data/processed


In [37]:
if not VALIDATION_PREDICTIONS_PATH.exists():
    matches = list(
        PROJECT_ROOT.rglob(
            "magnitude_validation_predictions.parquet"
        )
    )

    assert len(matches) == 1, (
        f"Expected one validation prediction file, found: {matches}"
    )

    VALIDATION_PREDICTIONS_PATH = matches[0]

if not MODEL_PANEL_PATH.exists():
    matches = list(
        PROJECT_ROOT.rglob(
            "magnitude_model_panel.parquet"
        )
    )

    assert len(matches) == 1, (
        f"Expected one magnitude model panel, found: {matches}"
    )

    MODEL_PANEL_PATH = matches[0]

print(
    "Validation predictions:",
    VALIDATION_PREDICTIONS_PATH,
)

print(
    "Model panel:",
    MODEL_PANEL_PATH,
)

Validation predictions: /Users/kushagr/Desktop/astra-assignment/data/processed/magnitude_validation_predictions.parquet
Model panel: /Users/kushagr/Desktop/astra-assignment/outputs/processed_data/magnitude_model_panel.parquet


In [38]:
magnitude_predictions_df = pd.read_parquet(
    VALIDATION_PREDICTIONS_PATH
)

magnitude_predictions_df["pred_date"] = pd.to_datetime(
    magnitude_predictions_df["pred_date"]
)

magnitude_predictions_df = (
    magnitude_predictions_df
    .sort_values(["pred_date", "symbol"])
    .reset_index(drop=True)
)

required_prediction_columns = [
    "symbol",
    "pred_date",
    "actual_magnitude_pct",
    "trailing_magnitude_20d",
    "pred_magnitude_pct",
]

missing_columns = [
    column
    for column in required_prediction_columns
    if column not in magnitude_predictions_df.columns
]

assert not missing_columns, missing_columns

print("Prediction shape:", magnitude_predictions_df.shape)
print(
    "Date range:",
    magnitude_predictions_df["pred_date"].min(),
    "to",
    magnitude_predictions_df["pred_date"].max(),
)

Prediction shape: (54009, 6)
Date range: 2024-04-08 00:00:00 to 2025-04-30 00:00:00


In [39]:
model_panel_df = pd.read_parquet(
    MODEL_PANEL_PATH
)

model_panel_df["pred_date"] = pd.to_datetime(
    model_panel_df["pred_date"]
)

assert not model_panel_df.duplicated(
    ["symbol", "pred_date"]
).any()

candidate_context_features = [
    "return_1d",
    "return_5d",
    "return_20d",
    "daily_volatility_20d",
    "overnight_volatility_20d",
    "short_term_volatility_5d",
    "gap_history_mean_20d",
    "gap_history_std_20d",
    "volume_zscore_20d",
    "volume_trend_5d_20d",
    "market_breadth",
    "aggregate_universe_volatility",
    "cross_sectional_return_dispersion",
    "day_of_week",
    "calendar_gap_days",
]

available_context_features = [
    column
    for column in candidate_context_features
    if column in model_panel_df.columns
]

print("Context features available:")
print(available_context_features)

Context features available:
['return_1d', 'return_5d', 'return_20d', 'daily_volatility_20d', 'volume_zscore_20d', 'volume_trend_5d_20d', 'market_breadth', 'aggregate_universe_volatility', 'cross_sectional_return_dispersion', 'day_of_week', 'calendar_gap_days']


In [40]:
context_reference_df = model_panel_df[
    [
        "symbol",
        "pred_date",
        *available_context_features,
    ]
].copy()

magnitude_confidence_df = (
    magnitude_predictions_df.merge(
        context_reference_df,
        on=["symbol", "pred_date"],
        how="left",
        validate="one_to_one",
    )
)

assert len(magnitude_confidence_df) == len(
    magnitude_predictions_df
)

print(
    "Magnitude-confidence dataset:",
    magnitude_confidence_df.shape,
)

Magnitude-confidence dataset: (54009, 17)


In [41]:
magnitude_confidence_df[
    "absolute_magnitude_error"
] = np.abs(
    magnitude_confidence_df["pred_magnitude_pct"]
    - magnitude_confidence_df["actual_magnitude_pct"]
)

magnitude_confidence_df[
    "log_absolute_magnitude_error"
] = np.log1p(
    magnitude_confidence_df[
        "absolute_magnitude_error"
    ]
)

magnitude_confidence_df[
    "prediction_vs_baseline_ratio"
] = (
    magnitude_confidence_df["pred_magnitude_pct"]
    / magnitude_confidence_df[
        "trailing_magnitude_20d"
    ].replace(0, np.nan)
)

magnitude_confidence_df[
    "prediction_baseline_difference"
] = (
    magnitude_confidence_df["pred_magnitude_pct"]
    - magnitude_confidence_df[
        "trailing_magnitude_20d"
    ]
)

magnitude_confidence_df[
    "log_predicted_magnitude"
] = np.log1p(
    magnitude_confidence_df[
        "pred_magnitude_pct"
    ].clip(lower=0)
)

MAGNITUDE_CONFIDENCE_FEATURES = [
    "pred_magnitude_pct",
    "trailing_magnitude_20d",
    "log_predicted_magnitude",
    "prediction_vs_baseline_ratio",
    "prediction_baseline_difference",
    *available_context_features,
]

MAGNITUDE_CONFIDENCE_FEATURES = list(
    dict.fromkeys(
        MAGNITUDE_CONFIDENCE_FEATURES
    )
)

print("Confidence feature count:", len(
    MAGNITUDE_CONFIDENCE_FEATURES
))

print(
    magnitude_confidence_df[
        "absolute_magnitude_error"
    ].describe()
)

Confidence feature count: 16
count    54009.000000
mean         0.465761
std          0.801321
min          0.000018
25%          0.148297
50%          0.297237
75%          0.474875
max         33.962155
Name: absolute_magnitude_error, dtype: float64


In [42]:
validation_dates = np.array(
    sorted(
        magnitude_confidence_df[
            "pred_date"
        ].drop_duplicates()
    )
)

confidence_cutoff_index = int(
    len(validation_dates) * 0.60
)

confidence_training_dates = set(
    validation_dates[:confidence_cutoff_index]
)

confidence_selection_dates = set(
    validation_dates[confidence_cutoff_index:]
)

confidence_train_df = (
    magnitude_confidence_df[
        magnitude_confidence_df["pred_date"].isin(
            confidence_training_dates
        )
    ]
    .copy()
)

confidence_selection_df = (
    magnitude_confidence_df[
        magnitude_confidence_df["pred_date"].isin(
            confidence_selection_dates
        )
    ]
    .copy()
)

print(
    "Confidence training:",
    confidence_train_df["pred_date"].min(),
    "to",
    confidence_train_df["pred_date"].max(),
    "| rows:",
    len(confidence_train_df),
)

print(
    "Confidence selection:",
    confidence_selection_df["pred_date"].min(),
    "to",
    confidence_selection_df["pred_date"].max(),
    "| rows:",
    len(confidence_selection_df),
)

Confidence training: 2024-04-08 00:00:00 to 2024-11-25 00:00:00 | rows: 31977
Confidence selection: 2024-11-26 00:00:00 to 2025-04-30 00:00:00 | rows: 22032


In [43]:
X_conf_train = confidence_train_df[
    MAGNITUDE_CONFIDENCE_FEATURES
].copy()

X_conf_selection = confidence_selection_df[
    MAGNITUDE_CONFIDENCE_FEATURES
].copy()

y_conf_train = confidence_train_df[
    "log_absolute_magnitude_error"
].copy()

fill_values = X_conf_train.median(
    numeric_only=True
)

X_conf_train = X_conf_train.fillna(
    fill_values
)

X_conf_selection = X_conf_selection.fillna(
    fill_values
)

X_conf_train = X_conf_train.replace(
    [np.inf, -np.inf],
    np.nan,
).fillna(fill_values)

X_conf_selection = X_conf_selection.replace(
    [np.inf, -np.inf],
    np.nan,
).fillna(fill_values)

print("Training matrix:", X_conf_train.shape)
print("Selection matrix:", X_conf_selection.shape)

Training matrix: (31977, 16)
Selection matrix: (22032, 16)


In [44]:
magnitude_confidence_model = lgb.LGBMRegressor(
    objective="regression_l1",
    n_estimators=750,
    num_leaves=15,
    learning_rate=0.03,
    min_child_samples=150,
    feature_fraction=0.80,
    bagging_fraction=0.80,
    bagging_freq=1,
    reg_lambda=2.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

magnitude_confidence_model.fit(
    X_conf_train,
    y_conf_train,
)

selection_expected_log_error = (
    magnitude_confidence_model.predict(
        X_conf_selection
    )
)

selection_expected_error = np.expm1(
    selection_expected_log_error
)

selection_expected_error = np.clip(
    selection_expected_error,
    0,
    None,
)

pd.Series(
    selection_expected_error,
    name="expected_absolute_error",
).describe()

count    22032.000000
mean         0.370729
std          0.184612
min          0.119254
25%          0.272466
50%          0.321557
75%          0.403553
max          1.871227
Name: expected_absolute_error, dtype: float64

In [53]:
large_conf_model = lgb.LGBMRegressor(
    objective="regression_l1",
    n_estimators=1500,
    num_leaves=31,
    learning_rate=0.03,
    min_child_samples=75,
    feature_fraction=0.90,
    bagging_fraction=0.80,
    bagging_freq=1,
    reg_lambda=2.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

large_conf_model.fit(
    X_conf_train,
    y_conf_train,
)

large_expected_log_error = (
    large_conf_model.predict(
        X_conf_selection
    )
)

large_expected_error = np.expm1(
    large_expected_log_error
)

large_expected_error = np.clip(
    large_expected_error,
    0,
    None,
)

large_training_expected_error = np.expm1(
    large_conf_model.predict(
        X_conf_train
    )
)

large_training_expected_error = np.clip(
    large_training_expected_error,
    0,
    None,
)

large_sorted_training_errors = np.sort(
    large_training_expected_error
)

large_confidence = expected_error_to_confidence(
    expected_error=large_expected_error,
    reference_errors=large_sorted_training_errors,
)

(
    large_metrics,
    large_deciles,
) = evaluate_magnitude_confidence(
    evaluation_df=confidence_selection_df,
    confidence=large_confidence,
)

pd.Series(
    large_metrics,
    name="large_tree_confidence",
)

confidence_error_spearman            0.285096
lowest_confidence_decile_mae         1.059935
highest_confidence_decile_mae        0.259063
confidence_gradient                  0.800872
relative_mae_improvement             0.755586
mean_confidence                      0.398657
n_obs                            22032.000000
Name: large_tree_confidence, dtype: float64

In [54]:
comparison = pd.DataFrame(
    [
        {
            "model": "current",
            **magnitude_confidence_metrics,
        },
        {
            "model": "large_tree",
            **large_metrics,
        },
    ]
)

comparison

,model,confidence_error_spearman,lowest_confidence_decile_mae,highest_confidence_decile_mae,confidence_gradient,relative_mae_improvement,mean_confidence,n_obs
0,current,0.299122,1.053069,0.260604,0.792465,0.752529,0.408144,22032
1,large_tree,0.285096,1.059935,0.259063,0.800872,0.755586,0.398657,22032


In [45]:
training_expected_error = np.expm1(
    magnitude_confidence_model.predict(
        X_conf_train
    )
)

training_expected_error = np.clip(
    training_expected_error,
    0,
    None,
)

sorted_training_expected_error = np.sort(
    training_expected_error
)

def expected_error_to_confidence(
    expected_error: np.ndarray,
    reference_errors: np.ndarray,
) -> np.ndarray:
    expected_error = np.asarray(
        expected_error,
        dtype=float,
    )

    reference_errors = np.asarray(
        reference_errors,
        dtype=float,
    )

    percentile = np.searchsorted(
        reference_errors,
        expected_error,
        side="right",
    ) / len(reference_errors)

    confidence = 1.0 - percentile

    return np.clip(
        confidence,
        0.0,
        1.0,
    )

selection_conf_magnitude = (
    expected_error_to_confidence(
        expected_error=selection_expected_error,
        reference_errors=(
            sorted_training_expected_error
        ),
    )
)

pd.Series(
    selection_conf_magnitude,
    name="conf_magnitude",
).describe()

count    22032.000000
mean         0.408144
std          0.275801
min          0.008881
25%          0.165611
50%          0.364684
75%          0.620852
max          1.000000
Name: conf_magnitude, dtype: float64

In [46]:
def evaluate_magnitude_confidence(
    evaluation_df: pd.DataFrame,
    confidence: np.ndarray,
) -> tuple[dict, pd.DataFrame]:
    result_df = evaluation_df[
        [
            "symbol",
            "pred_date",
            "actual_magnitude_pct",
            "pred_magnitude_pct",
            "absolute_magnitude_error",
        ]
    ].copy()

    result_df["conf_magnitude"] = np.asarray(
        confidence,
        dtype=float,
    )

    valid_mask = (
        np.isfinite(
            result_df["absolute_magnitude_error"]
        )
        & np.isfinite(
            result_df["conf_magnitude"]
        )
    )

    result_df = result_df.loc[
        valid_mask
    ].copy()

    error_confidence_spearman = spearmanr(
        result_df["conf_magnitude"],
        -result_df["absolute_magnitude_error"],
    ).statistic

    result_df["confidence_decile"] = pd.qcut(
        result_df["conf_magnitude"],
        q=10,
        labels=False,
        duplicates="drop",
    )

    decile_summary = (
        result_df.groupby(
            "confidence_decile",
            as_index=False,
        )
        .agg(
            mean_confidence=(
                "conf_magnitude",
                "mean",
            ),
            mae=(
                "absolute_magnitude_error",
                "mean",
            ),
            median_absolute_error=(
                "absolute_magnitude_error",
                "median",
            ),
            observations=(
                "absolute_magnitude_error",
                "size",
            ),
        )
        .sort_values("confidence_decile")
        .reset_index(drop=True)
    )

    lowest_confidence_mae = (
        decile_summary.iloc[0]["mae"]
    )

    highest_confidence_mae = (
        decile_summary.iloc[-1]["mae"]
    )

    confidence_gradient = (
        lowest_confidence_mae
        - highest_confidence_mae
    )

    relative_mae_improvement = (
        confidence_gradient
        / lowest_confidence_mae
        if lowest_confidence_mae > 0
        else np.nan
    )

    metrics = {
        "confidence_error_spearman": float(
            error_confidence_spearman
        ),
        "lowest_confidence_decile_mae": float(
            lowest_confidence_mae
        ),
        "highest_confidence_decile_mae": float(
            highest_confidence_mae
        ),
        "confidence_gradient": float(
            confidence_gradient
        ),
        "relative_mae_improvement": float(
            relative_mae_improvement
        ),
        "mean_confidence": float(
            result_df["conf_magnitude"].mean()
        ),
        "n_obs": int(len(result_df)),
    }

    return metrics, decile_summary

In [47]:
(
    magnitude_confidence_metrics,
    magnitude_confidence_deciles,
) = evaluate_magnitude_confidence(
    evaluation_df=confidence_selection_df,
    confidence=selection_conf_magnitude,
)

print("Magnitude confidence metrics:")
print(
    pd.Series(
        magnitude_confidence_metrics
    )
)

magnitude_confidence_deciles

Magnitude confidence metrics:
confidence_error_spearman            0.299122
lowest_confidence_decile_mae         1.053069
highest_confidence_decile_mae        0.260604
confidence_gradient                  0.792465
relative_mae_improvement             0.752529
mean_confidence                      0.408144
n_obs                            22032.000000
dtype: float64


,confidence_decile,mean_confidence,mae,median_absolute_error,observations
0,0,0.042506,1.053069,0.500555,2204
1,1,0.099019,0.814763,0.446191,2205
2,2,0.166205,0.551041,0.362369,2201
3,3,0.242264,0.455692,0.337166,2203
4,4,0.322067,0.413614,0.307178,2203
5,5,0.412112,0.390450,0.306246,2204
6,6,0.510707,0.373841,0.290123,2203
7,7,0.621452,0.333456,0.266258,2203
8,8,0.754117,0.293689,0.245454,2203
9,9,0.911222,0.260604,0.213953,2203


In [48]:
FINAL_MAGNITUDE_CONFIDENCE_MODEL = (
    "lightgbm_expected_error_model"
)

final_magnitude_confidence_decision = {
    "selected_model": FINAL_MAGNITUDE_CONFIDENCE_MODEL,
    "confidence_error_spearman": float(
        magnitude_confidence_metrics[
            "confidence_error_spearman"
        ]
    ),
    "confidence_gradient": float(
        magnitude_confidence_metrics[
            "confidence_gradient"
        ]
    ),
    "relative_mae_improvement": float(
        magnitude_confidence_metrics[
            "relative_mae_improvement"
        ]
    ),
    "selection_split": "late_validation",
    "test_used": False,
}

final_magnitude_confidence_decision

{'selected_model': 'lightgbm_expected_error_model',
 'confidence_error_spearman': 0.2991217273189215,
 'confidence_gradient': 0.7924649788384409,
 'relative_mae_improvement': 0.7525286866239927,
 'selection_split': 'late_validation',
 'test_used': False}

In [49]:
confidence_selection_df = (
    confidence_selection_df.copy()
)

confidence_selection_df[
    "expected_absolute_error"
] = selection_expected_error

confidence_selection_df[
    "conf_magnitude"
] = selection_conf_magnitude

MAGNITUDE_CONFIDENCE_VALIDATION_PATH = (
    PROCESSED_DATA_DIR
    / "magnitude_confidence_validation.parquet"
)

confidence_selection_df.to_parquet(
    MAGNITUDE_CONFIDENCE_VALIDATION_PATH,
    index=False,
)

print(
    "Saved:",
    MAGNITUDE_CONFIDENCE_VALIDATION_PATH
)

Saved: /Users/kushagr/Desktop/astra-assignment/data/processed/magnitude_confidence_validation.parquet


In [50]:
MAGNITUDE_CONFIDENCE_RESULTS_PATH = (
    PROCESSED_DATA_DIR
    / "magnitude_confidence_results.csv"
)

pd.Series(
    magnitude_confidence_metrics
).to_csv(
    MAGNITUDE_CONFIDENCE_RESULTS_PATH,
    header=False,
)

print(
    "Saved:",
    MAGNITUDE_CONFIDENCE_RESULTS_PATH
)

Saved: /Users/kushagr/Desktop/astra-assignment/data/processed/magnitude_confidence_results.csv


In [51]:
import json

MAGNITUDE_CONFIDENCE_CONFIG_PATH = (
    PROCESSED_DATA_DIR
    / "final_magnitude_confidence_config.json"
)

config = {
    "model": "lightgbm_expected_error_model",
    "metrics": {
        key: (
            float(value)
            if isinstance(value, (float, np.floating))
            else int(value)
        )
        for key, value in magnitude_confidence_metrics.items()
    },
    "selection_split": "late_validation",
    "test_used": False,
}

with open(
    MAGNITUDE_CONFIDENCE_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        config,
        f,
        indent=2,
    )

print(
    "Saved:",
    MAGNITUDE_CONFIDENCE_CONFIG_PATH
)

Saved: /Users/kushagr/Desktop/astra-assignment/data/processed/final_magnitude_confidence_config.json


In [52]:
print("Magnitude confidence notebook complete.\n")

print(pd.Series(
    magnitude_confidence_metrics
))

print("\nSelected model:")
print("LightGBM Expected Error Model")


Magnitude confidence notebook complete.

confidence_error_spearman            0.299122
lowest_confidence_decile_mae         1.053069
highest_confidence_decile_mae        0.260604
confidence_gradient                  0.792465
relative_mae_improvement             0.752529
mean_confidence                      0.408144
n_obs                            22032.000000
dtype: float64

Selected model:
LightGBM Expected Error Model
